# StreamDiffusion Demo (Extended): Real-Time Diffusion with Video Processing

This notebook extends the base StreamDiffusion demo with:
- Advanced batch image-to-image processing
- Video-to-video pipeline (frame extraction, batch transformation, reassembly)
- OBS live stream processing
- YouTube video downloading and processing
- YouTube live stream processing with buffered playback

**Platform note**: Several cells use `google.colab` APIs (file upload/download, `cv2_imshow`). These are marked as Colab-specific and will need adaptation for other environments.

## Environment Setup

In [ ]:
!pip install torch==2.1.0 torchvision==0.16.0 xformers --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!git clone https://github.com/cumulo-autumn/StreamDiffusion.git

In [ ]:
%cd StreamDiffusion

In [ ]:
!python setup.py develop easy_install streamdiffusion[tensorrt]

In [ ]:
!python -m streamdiffusion.tools.install-tensorrt

In [ ]:
import os

if os.getenv("COLAB_RELEASE_TAG"):
    %cd /content/StreamDiffusion
else:
    # For non-Colab environments, change to the StreamDiffusion directory
    # relative to the current working directory (assumes it was cloned above).
    %cd StreamDiffusion

## Text to Image Generation

In [ ]:
!pip install -e .[tensorrt]

In [ ]:
!pip install --upgrade huggingface_hub
!pip install --upgrade diffusers

In [ ]:
from utils.wrapper import StreamDiffusionWrapper

In [ ]:
stream = StreamDiffusionWrapper(
    model_id_or_path = "KBlueLeaf/kohaku-v2.1",
    lora_dict = None,
    t_index_list = [0, 16, 32, 45],
    frame_buffer_size = 1,
    width = 512,
    height = 512,
    warmup = 10,
    acceleration = "xformers",
    mode = "txt2img",
    use_denoising_batch = False,
    cfg_type = "none",
    seed = 2
)

In [ ]:
prompt = "A girl with brown hair, smiling, and wearing a shirt"

In [ ]:
stream.prepare(
    prompt = prompt,
    num_inference_steps = 50
)

In [ ]:
output_image = stream()
output_image.save("images/outputs/output.png")

## Multi Text-to-image

In [ ]:
stream = StreamDiffusionWrapper(
    model_id_or_path = "KBlueLeaf/kohaku-v2.1",
    lora_dict = None,
    t_index_list = [0, 16, 32, 45],
    frame_buffer_size = 3,
    width = 512,
    height = 512,
    warmup = 10,
    acceleration = "xformers",
    mode = "txt2img",
    use_denoising_batch = False,
    cfg_type = "none",
    seed = 2
)

In [ ]:
new_prompt = "A girl with brown hair, smiling, and wearing a shirt"

In [ ]:
stream.prepare(
    prompt = new_prompt,
    num_inference_steps = 50
)

In [ ]:
output_images = stream()
for i, output_image in enumerate(output_images):
    output_image.save(f"images/outputs/output_{i}.png")

## Text to image without Wrapper

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained("KBlueLeaf/kohaku-v2.1").to(
    device = torch.device("cuda"),
    dtype=torch.float16
)

In [ ]:
from diffusers import AutoencoderTiny
from streamdiffusion import StreamDiffusion
from streamdiffusion.image_utils import postprocess_image

In [ ]:
stream = StreamDiffusion(
    pipe,
    t_index_list = [0, 16, 32, 45],
    torch_dtype=torch.float16,
    cfg_type = "none"
)

In [ ]:
stream.load_lcm_lora()
stream.fuse_lora()

In [ ]:
stream.vae = AutoencoderTiny.from_pretrained("madebyollin/taesd").to(
    device = pipe.device,
    dtype=pipe.dtype
)

In [ ]:
pipe.enable_xformers_memory_efficient_attention()

In [ ]:
stream.prepare(prompt)

In [ ]:
for _ in range(4):
  stream()

In [ ]:
x_output = stream.txt2img()

In [ ]:
output_image = postprocess_image(x_output, output_type="pil")[0]

In [ ]:
output_image.save("images/outputs/output.png")

## Img to Img Generation

In [ ]:
stream = StreamDiffusionWrapper(
    model_id_or_path = "KBlueLeaf/kohaku-v2.1",
    lora_dict = None,
    t_index_list = [22, 32, 45],
    frame_buffer_size = 1,
    width = 512,
    height = 512,
    warmup = 10,
    acceleration = "xformers",
    mode = "img2img",
    use_denoising_batch = True,
    cfg_type = "self",
    seed = 2
)

In [ ]:
prompt = "mantain the same image"
negative_prompt = "low quality, blurry, low resolution"
image = "images/outputs/output.png"

In [ ]:
stream.prepare(
    prompt = prompt,
    negative_prompt = negative_prompt,
    num_inference_steps = 50,
    guidance_scale = 1.2,
    delta = 0.5
)

In [ ]:
image_tensor = stream.preprocess_image(image)

In [ ]:
for _ in range(stream.batch_size - 1):
  stream(image=image)

In [ ]:
output_image = stream(image = image_tensor)

In [ ]:
output_image.save("images/outputs/output8.png")

## Advanced Batch Processing

Low-level img2img pipeline with custom `t_index_list` and resolution scaling, followed by a modified `multi.py` script for batch image-to-image transformation.

In [ ]:
import torch
from diffusers import AutoencoderTiny, StableDiffusionPipeline
from diffusers.utils import load_image

from streamdiffusion import StreamDiffusion
from streamdiffusion.image_utils import postprocess_image

# You can load any models using diffuser's StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained("KBlueLeaf/kohaku-v2.1").to(
    device=torch.device("cuda"),
    dtype=torch.float16,
)


#KBlueLeaf/kohaku-v2.1

# Wrap the pipeline in StreamDiffusion
stream = StreamDiffusion(
    pipe,
    #t_index_list=[32, 45],
    t_index_list=[36, 46],
    torch_dtype=torch.float16,
)

# If the loaded model is not LCM, merge LCM #KBlueLeaf/kohaku-v2.1
stream.load_lcm_lora()
stream.fuse_lora()


# Use Tiny VAE for further acceleration
stream.vae = AutoencoderTiny.from_pretrained("madebyollin/taesd").to(device=pipe.device, dtype=pipe.dtype)
# Enable acceleration
pipe.enable_xformers_memory_efficient_attention()


prompt = "tsubasa captain soccer players"
# Prepare the stream
stream.prepare(prompt)

# Prepare image
init_image = load_image("images/outputs/frame_00019.png").resize((512, 512))

# Warmup >= len(t_index_list) x frame_buffer_size
for _ in range(2):
    stream(init_image)

# Run the stream with a maximum iteration limit to avoid infinite loops
MAX_ITERATIONS = 100
for iteration in range(MAX_ITERATIONS):
    x_output = stream(init_image)
    postprocess_image(x_output, output_type="pil")[0].show()
    postprocess_image(x_output, output_type="pil")[0].save("images/outputs/output8.png")
    input_response = input(f"[{iteration+1}/{MAX_ITERATIONS}] Press Enter to continue or type 'stop' to exit: ")
    if input_response == "stop":
        break
else:
    print(f"Reached maximum iterations ({MAX_ITERATIONS}). Stopping.")

### Modified multi.py

Custom version of `examples/img2img/multi.py` with adjusted `t_index_list`, doubled resolution (`width*2`, `height*2`), and increased guidance scale.

In [ ]:
import glob
import os
import sys
from typing import Literal, Dict, Optional

import fire


sys.path.append(os.path.join(os.path.dirname(__file__), "..", ".."))

from utils.wrapper import StreamDiffusionWrapper

CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))


def main(
    input: str = os.path.join(CURRENT_DIR, "..", "..", "images", "inputs"),
    output: str = os.path.join(CURRENT_DIR, "..", "..", "images", "outputs"),
    model_id_or_path: str = "KBlueLeaf/kohaku-v2.1",
    lora_dict: Optional[Dict[str, float]] = None,
    prompt: str = "1girl with brown dog hair, thick glasses, smiling",
    negative_prompt: str = "low quality, bad quality, blurry, low resolution",
    width: int = 512,
    height: int = 512,
    acceleration: Literal["none", "xformers", "tensorrt"] = "xformers",
    use_denoising_batch: bool = True,
    guidance_scale: float = 1.2,
    cfg_type: Literal["none", "full", "self", "initialize"] = "self",
    seed: int = 2,
    delta: float = 0.5,
):
    """
    Initializes the StreamDiffusionWrapper.

    Parameters
    ----------
    input : str, optional
        The input directory to load images from.
    output : str, optional
        The output directory to save images to.
    model_id_or_path : str
        The model id or path to load.
    lora_dict : Optional[Dict[str, float]], optional
        The lora_dict to load, by default None.
        Keys are the LoRA names and values are the LoRA scales.
        Example: {'LoRA_1' : 0.5 , 'LoRA_2' : 0.7 ,...}
    prompt : str
        The prompt to generate images from.
    negative_prompt : str, optional
        The negative prompt to use.
    width : int, optional
        The width of the image, by default 512.
    height : int, optional
        The height of the image, by default 512.
    acceleration : Literal["none", "xformers", "tensorrt"], optional
        The acceleration method, by default "tensorrt".
    use_denoising_batch : bool, optional
        Whether to use denoising batch or not, by default True.
    guidance_scale : float, optional
        The CFG scale, by default 1.2.
    cfg_type : Literal["none", "full", "self", "initialize"],
    optional
        The cfg_type for img2img mode, by default "self".
        You cannot use anything other than "none" for txt2img mode.
    seed : int, optional
        The seed, by default 2. if -1, use random seed.
    delta : float, optional
        The delta multiplier of virtual residual noise,
        by default 1.0.
    """

    if not os.path.exists(output):
        os.makedirs(output, exist_ok=True)

    if guidance_scale <= 1.0:
        cfg_type = "none"

    stream = StreamDiffusionWrapper(
        model_id_or_path=model_id_or_path,
        lora_dict=lora_dict,
        #t_index_list=[32, 40, 45],
        t_index_list=[40, 46, 49], # here
        frame_buffer_size=1,
        width=width*2, # here
        height=height*2, # here
        warmup=10,
        acceleration=acceleration,
        mode="img2img",
        use_denoising_batch=use_denoising_batch,
        cfg_type=cfg_type,
        seed=seed,
    )

    stream.prepare(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=50,
        guidance_scale=guidance_scale*1.3, # here
        delta=delta,
    )

    images = glob.glob(os.path.join(input, "*"))
    images = images + [images[-1]] * (stream.batch_size - 1)
    outputs = []

    for i in range(stream.batch_size - 1):
        image = images.pop(0)
        outputs.append(image)
        output_image = stream(image=image)

    for image in images:
        outputs.append(image)
        try:
            output_image = stream(image=image)
        except Exception:
            continue

        name = outputs.pop(0)
        basename = os.path.splitext(os.path.basename(name))[0]
        output_image.save(os.path.join(output, f"{basename}.png"))


if __name__ == "__main__":
    fire.Fire(main)


## Video Utilities

Helper functions for converting between video files and image frame sequences.

### Create Video from Images

In [ ]:
import cv2
import os


def create_video_from_images(image_dir, output_video, fps=30):
    images = sorted(
        [img for img in os.listdir(image_dir) if img.endswith(".png")]
    )
    print(images)
    frame = cv2.imread(os.path.join(image_dir, images[0]))
    height, width, _ = frame.shape

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    for image in images:
        print(image)
        frame = cv2.imread(os.path.join(image_dir, image))
        video.write(frame)

    video.release()
    print(f"Video created at {output_video}")

In [ ]:
create_video_from_images("images/outputs/ig18", "images/outputs/ig18_vid/output_video.mp4", fps=5)

### Extract Frames from Video

In [ ]:
import cv2
import os

def extract_frames(video_path, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    cap = cv2.VideoCapture(video_path)
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.imwrite(os.path.join(output_dir, f"frame_{frame_count:04d}.png"), frame)
        frame_count += 1

    cap.release()
    print(f"Extracted {frame_count} frames to {output_dir}")

In [ ]:
extract_frames("images/outputs/soccer_video/soccer_example_cut.mp4", "/content/StreamDiffusion/images/outputs/soccer_video/soccer_example_imgs")

## Video-to-Video Pipeline

End-to-end pipeline: extract frames from a video, process them in batches using StreamDiffusion's img2img, and reassemble into an output video.

**Platform note (Google Colab)**: Uses `google.colab.files` for downloading results. Replace with local file I/O on other platforms.

In [ ]:
!pip install -r examples/vid2vid/requirements.txt

In [ ]:
!python examples/vid2vid/main.py --input images/outputs/soccer_example_cut_2.mp4 --output images/outputs/soccer_example_out.mp4

In [ ]:
!python examples/img2img/multi.py --input images/outputs/images_soccer_cut --output images/outputs/ig18 --prompt "tsubasa captain soccer players with the ball" --negative_prompt "tsubasa captain soccer players with the ball"

### Full Video Processing Pipeline

Self-contained pipeline that extracts frames, processes them in batches, creates an output video, and downloads it.

In [ ]:
import os
import cv2
import subprocess

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Function to extract frames from a video
def extract_frames(video_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.imwrite(os.path.join(output_dir, f"frame_{frame_count:04d}.png"), frame)
        frame_count += 1

    cap.release()
    print(f"Extracted {frame_count} frames to {output_dir}")
    return frame_count

def process_images_with_script(input_dir, output_dir, batch_size, script_path, prompt, negative_prompt):
    os.makedirs(output_dir, exist_ok=True)
    images = sorted([img for img in os.listdir(input_dir) if img.endswith(".png")])
    total_images = len(images)

    for i in range(0, total_images, batch_size):
        batch_dir = os.path.join(input_dir, f"batch_{i // batch_size}")
        os.makedirs(batch_dir, exist_ok=True)
        batch = images[i:i+batch_size]

        # Move batch images to a temporary folder
        for image_name in batch:
            image_path = os.path.join(input_dir, image_name)
            os.rename(image_path, os.path.join(batch_dir, image_name))

        # Construct the command
        command = [
            "python", script_path,
            "--input", batch_dir,
            "--output", output_dir,
            "--prompt", prompt,
            "--negative_prompt", negative_prompt
        ]
        print(f"Processing batch {i // batch_size + 1}/{(total_images - 1) // batch_size + 1}")

        # Run the command and capture logs
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        # Stream logs in real-time
        for line in process.stdout:
            print(line, end="")

        # Wait for process to complete and handle errors
        process.wait()
        if process.returncode != 0:
            print(f"Error in batch {i // batch_size + 1}:")
            print(process.stderr.read())
            break

    print(f"Processed all images. Transformed images saved to {output_dir}")

# Function to create a video from images
def create_video_from_images(image_dir, output_video, fps=30):
    images = sorted([img for img in os.listdir(image_dir) if img.endswith(".png")])
    frame = cv2.imread(os.path.join(image_dir, images[0]))
    height, width, _ = frame.shape

    os.makedirs(os.path.dirname(output_video), exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    for image in images:
        frame = cv2.imread(os.path.join(image_dir, image))
        video.write(frame)

    video.release()
    print(f"Video created at {output_video}")

# Paths and configuration
video_path = "images/outputs/soccer_video/soccer_example_cut.mp4"  # Upload your video file
extracted_frames_dir = "images/outputs/content/frames"
transformed_frames_dir = "images/outputs/content/transformed_frames"
output_video_path = "images/outputs/content/output_video.mp4"
batch_size = 10
script_path = "examples/img2img/multi.py"
prompt = "tsubasa captain soccer players with the ball"
negative_prompt = "tsubasa captain soccer players with the ball"

# Step 1: Extract frames from the video
print("Extracting frames...")
total_frames = extract_frames(video_path, extracted_frames_dir)

# Step 2: Process frames in batches using your script
print("Processing frames in batches...")
process_images_with_script(
    input_dir=extracted_frames_dir,
    output_dir=transformed_frames_dir,
    batch_size=batch_size,
    script_path=script_path,
    prompt=prompt,
    negative_prompt=negative_prompt,
)

# Step 3: Convert transformed frames back into a video
print("Creating video from transformed frames...")
create_video_from_images(transformed_frames_dir, output_video_path, fps=30)

# Step 4: Download the resulting video
print("Downloading the final video...")
if IN_COLAB:
    files.download(output_video_path)
else:
    print(f"Video saved to: {output_video_path}")

## OBS Streaming

Captures frames from an OBS RTMP stream, applies StreamDiffusion transformation frame-by-frame, and outputs the transformed stream via RTMP.

Requires OBS Studio configured to stream to `rtmp://127.0.0.1/live`.

In [ ]:
import cv2
import numpy as np
import subprocess
import os
import threading

# Parameters
obs_stream_url = "rtmp://127.0.0.1/live"  # Replace with OBS stream URL
output_fps = 30
output_size = (640, 360)  # Adjust as per your needs
output_video_stream = "rtmp://127.0.0.1/transformed"  # Replace with output stream URL
script_path = "examples/img2img/multi.py"
prompt = "tsubasa captain soccer players with the ball"
negative_prompt = "tsubasa captain soccer players with the ball"

# Shutdown flag for clean termination
shutdown_event = threading.Event()

# Transform Image Function
def transform_image(input_image, temp_input="temp_input.png", temp_output="temp_output.png"):
    """
    Saves the input image to disk, processes it using the script, and loads the transformed image.
    """
    cv2.imwrite(temp_input, input_image)
    command = [
        "python", script_path,
        "--input", temp_input,
        "--output", temp_output,
        "--prompt", prompt,
        "--negative_prompt", negative_prompt
    ]
    process = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if process.returncode != 0:
        print(f"Error in transformation: {process.stderr.decode('utf-8')}")
        return input_image  # Return original frame on failure
    if os.path.exists(temp_output):
        return cv2.imread(temp_output)
    return input_image

# Stream Input Frames, Transform, and Output
def process_stream():
    cap = None
    out = None
    try:
        cap = cv2.VideoCapture(obs_stream_url)
        if not cap.isOpened():
            print("Error: Unable to open OBS stream.")
            return

        # Video Writer for output stream
        fourcc = cv2.VideoWriter_fourcc(*"X264")
        out = cv2.VideoWriter(
            output_video_stream, fourcc, output_fps, output_size, True
        )

        print("Streaming from OBS and applying transformations...")

        while not shutdown_event.is_set():
            ret, frame = cap.read()
            if not ret:
                print("Stream ended or interrupted.")
                break

            # Resize for consistent processing
            frame = cv2.resize(frame, output_size)

            # Transform the frame
            transformed_frame = transform_image(frame)

            # Write to the output stream
            out.write(transformed_frame)

            # Show the frame locally (optional)
            cv2.imshow("Transformed Stream", transformed_frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        # Properly release all resources
        print("Cleaning up streaming resources...")
        if cap is not None:
            cap.release()
        if out is not None:
            out.release()
        cv2.destroyAllWindows()
        # Clean up temp files
        for temp_file in ["temp_input.png", "temp_output.png"]:
            if os.path.exists(temp_file):
                os.remove(temp_file)
        print("Streaming resources released.")

def stop_stream():
    """Call this function to gracefully stop the stream."""
    shutdown_event.set()

# Run the Stream Processor
if __name__ == "__main__":
    stream_thread = threading.Thread(target=process_stream)
    stream_thread.start()
    # To stop gracefully, call stop_stream() or press 'q' in the display window

## YouTube Video Processing

Downloads a YouTube video, extracts frames, processes them through StreamDiffusion, and reassembles the result.

**Platform note (Google Colab)**: Uses `google.colab.files` for downloading results.

In [ ]:
# !pip install yt-dlp  # Uncomment if yt-dlp is not installed

In [ ]:
import os
import cv2
import subprocess

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Function to download YouTube video using yt-dlp
def download_youtube_video(youtube_url, output_path):
    command = ["yt-dlp", "-f", "best[ext=mp4]", "-o", output_path, youtube_url]
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode != 0:
        raise RuntimeError(f"Failed to download video: {result.stderr.decode('utf-8')}")
    print(f"Downloaded video to {output_path}")

# Function to extract frames from a video
def extract_frames(video_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.imwrite(os.path.join(output_dir, f"frame_{frame_count:04d}.png"), frame)
        frame_count += 1

    cap.release()
    print(f"Extracted {frame_count} frames to {output_dir}")
    return frame_count

# Function to process images using a transformation script
def process_images_with_script(input_dir, output_dir, batch_size, script_path, prompt, negative_prompt):
    os.makedirs(output_dir, exist_ok=True)
    images = sorted([img for img in os.listdir(input_dir) if img.endswith(".png")])
    total_images = len(images)

    for i in range(0, total_images, batch_size):
        batch_dir = os.path.join(input_dir, f"batch_{i // batch_size}")
        os.makedirs(batch_dir, exist_ok=True)
        batch = images[i:i+batch_size]

        # Move batch images to a temporary folder
        for image_name in batch:
            image_path = os.path.join(input_dir, image_name)
            os.rename(image_path, os.path.join(batch_dir, image_name))

        # Construct the command
        command = [
            "python", script_path,
            "--input", batch_dir,
            "--output", output_dir,
            "--prompt", prompt,
            "--negative_prompt", negative_prompt
        ]
        print(f"Processing batch {i // batch_size + 1}/{(total_images - 1) // batch_size + 1}")

        # Run the command and capture logs
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        # Stream logs in real-time
        for line in process.stdout:
            print(line, end="")

        # Wait for process to complete and handle errors
        process.wait()
        if process.returncode != 0:
            print(f"Error in batch {i // batch_size + 1}:")
            print(process.stderr.read())
            break

    print(f"Processed all images. Transformed images saved to {output_dir}")

# Function to create a video from images
def create_video_from_images(image_dir, output_video, fps=30):
    images = sorted([img for img in os.listdir(image_dir) if img.endswith(".png")])
    frame = cv2.imread(os.path.join(image_dir, images[0]))
    height, width, _ = frame.shape

    os.makedirs(os.path.dirname(output_video), exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    for image in images:
        frame = cv2.imread(os.path.join(image_dir, image))
        video.write(frame)

    video.release()
    print(f"Video created at {output_video}")

# Paths and configuration
youtube_url = "https://www.youtube.com/watch?v=ZEFLcPjxw6w"
video_path = "input_video.mp4"
extracted_frames_dir = "images/outputs/content/frames"
transformed_frames_dir = "images/outputs/content/transformed_frames"
output_video_path = "images/outputs/content/output_video.mp4"
batch_size = 10
script_path = "examples/img2img/multi.py"
prompt = "tsubasa captain soccer players with the ball"
negative_prompt = "tsubasa captain soccer players with the ball"

# Step 1: Download the YouTube video
print("Downloading YouTube video...")
download_youtube_video(youtube_url, video_path)

# Step 2: Extract frames from the video
print("Extracting frames...")
total_frames = extract_frames(video_path, extracted_frames_dir)

# Step 3: Process frames in batches using your script
print("Processing frames in batches...")
process_images_with_script(
    input_dir=extracted_frames_dir,
    output_dir=transformed_frames_dir,
    batch_size=batch_size,
    script_path=script_path,
    prompt=prompt,
    negative_prompt=negative_prompt,
)

# Step 4: Convert transformed frames back into a video
print("Creating video from transformed frames...")
create_video_from_images(transformed_frames_dir, output_video_path, fps=30)

# Step 5: Download the resulting video
print("Downloading the final video...")
if IN_COLAB:
    files.download(output_video_path)
else:
    print(f"Video saved to: {output_video_path}")

In [ ]:
# Step 4: Convert transformed frames back into a video
print("Creating video from transformed frames...")
create_video_from_images(transformed_frames_dir, output_video_path, fps=30)

# Step 5: Download the resulting video
print("Downloading the final video...")
try:
    from google.colab import files
    files.download(output_video_path)
except ImportError:
    print(f"Video saved to: {output_video_path}")

## YouTube Live Streaming

Captures frames from a YouTube live stream, processes them in batches through StreamDiffusion, and displays the transformed output with buffered playback.

Uses a producer-consumer architecture with separate threads for:
1. Frame capture (with frame skipping to match target FPS)
2. Batch transformation via the img2img script
3. Buffered display of transformed frames

**Platform note (Google Colab)**: Uses `google.colab.patches.cv2_imshow` and `IPython.display.clear_output` for in-notebook display.

In [ ]:
import cv2
import os
import queue
import threading
import subprocess
import time
from collections import deque

try:
    from google.colab.patches import cv2_imshow
    from IPython.display import clear_output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Parameters
youtube_url = "https://www.youtube.com/watch?v=yrXEJ6H5hDM"
script_path = "examples/img2img/multi.py"
prompt = "tsubasa captain soccer players with the ball"
negative_prompt = "tsubasa captain soccer players with the ball"
temp_input_dir = "input_frames"
temp_output_dir = "output_frames"
output_fps = 15
input_queue = queue.Queue(maxsize=50)  # Queue for unprocessed frames
output_buffer = deque(maxlen=50)  # Buffer for transformed frames
buffer_size_threshold = 100  # Number of frames to accumulate before streaming
delay_time = 5  # Seconds of delay added for buffering
shutdown_event = threading.Event()
os.makedirs(temp_input_dir, exist_ok=True)
os.makedirs(temp_output_dir, exist_ok=True)

# Step 1: Extract YouTube Live Stream URL
def get_stream_url(youtube_url):
    command = ["yt-dlp", "-f", "best[ext=mp4]", "--get-url", youtube_url]
    result = subprocess.run(command, stdout=subprocess.PIPE, text=True)
    if result.returncode != 0:
        raise RuntimeError("Failed to fetch the YouTube stream URL.")
    return result.stdout.strip()

# Step 2: Capture Frames with Frame Skipping
def capture_frames(stream_url, target_fps=10):
    cap = cv2.VideoCapture(stream_url)
    if not cap.isOpened():
        print("Error: Unable to open the live stream.")
        return

    print("Capturing frames...")
    actual_fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
    frame_interval = max(1, actual_fps // target_fps)
    frame_count = 0

    try:
        while not shutdown_event.is_set():
            ret, frame = cap.read()
            if not ret:
                print("Stream ended or interrupted.")
                break

            # Skip frames to match the target FPS
            if frame_count % frame_interval == 0:
                if not input_queue.full():
                    input_queue.put((frame_count, frame))
                else:
                    print("Input queue is full; dropping frame.")

            frame_count += 1
    finally:
        cap.release()
        print("Capture resources released.")

# Step 3: Transform Frames in Batches
def transform_frames(batch_size=10):
    while not shutdown_event.is_set():
        try:
            batch = []
            for _ in range(batch_size):
                frame_count, frame = input_queue.get(timeout=5)
                input_path = os.path.join(temp_input_dir, f"frame_{frame_count:04d}.png")
                cv2.imwrite(input_path, frame)
                batch.append((frame_count, input_path))

            # Run transformation script on the batch
            command = [
                "python", script_path,
                "--input", temp_input_dir,
                "--output", temp_output_dir,
                "--prompt", prompt,
                "--negative_prompt", negative_prompt
            ]
            subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

            # Load transformed frames and add them to the output buffer
            for frame_count, input_path in batch:
                output_path = os.path.join(temp_output_dir, f"frame_{frame_count:04d}.png")
                if os.path.exists(output_path):
                    transformed_frame = cv2.imread(output_path)
                    output_buffer.append((frame_count, transformed_frame))
                else:
                    print(f"Transformation failed for frame {frame_count}; using original.")
                    output_buffer.append((frame_count, frame))

                # Clean up
                if os.path.exists(input_path):
                    os.remove(input_path)
                if os.path.exists(output_path):
                    os.remove(output_path)

        except queue.Empty:
            print("No frames to transform; waiting...")

# Step 4: Stream Buffered Frames
def stream_buffered_frames():
    while not shutdown_event.is_set():
        if len(output_buffer) >= buffer_size_threshold:
            print(f"Streaming {buffer_size_threshold} buffered frames...")
            for _ in range(buffer_size_threshold):
                if output_buffer:
                    frame_count, frame = output_buffer.popleft()
                    if IN_COLAB:
                        clear_output(wait=True)
                        cv2_imshow(cv2.resize(frame, (640, 360)))
                    else:
                        cv2.imshow("Transformed Stream", cv2.resize(frame, (640, 360)))
                        if cv2.waitKey(1) & 0xFF == ord("q"):
                            shutdown_event.set()
                            return

            time.sleep(delay_time)  # Add artificial delay to simulate buffering
        else:
            print("Buffer underflow; waiting for more frames...")
            time.sleep(1)

def stop_streaming():
    """Call this function to gracefully stop all streaming threads."""
    shutdown_event.set()
    print("Shutdown signal sent to all threads.")

# Step 5: Main Function
def main():
    try:
        # Get YouTube live stream URL
        stream_url = get_stream_url(youtube_url)
        print(f"Stream URL: {stream_url}")

        # Start threads for capturing, transforming, and streaming
        capture_thread = threading.Thread(target=capture_frames, args=(stream_url,), daemon=True)
        transform_thread = threading.Thread(target=transform_frames, daemon=True)
        stream_thread = threading.Thread(target=stream_buffered_frames, daemon=True)

        capture_thread.start()
        transform_thread.start()
        stream_thread.start()

        capture_thread.join()
        transform_thread.join()
        stream_thread.join()

    except KeyboardInterrupt:
        print("Interrupted by user. Shutting down...")
        shutdown_event.set()
    except Exception as e:
        print(f"Error: {e}")
        shutdown_event.set()
    finally:
        if not IN_COLAB:
            cv2.destroyAllWindows()
        print("All resources cleaned up.")

# Run the Script
if __name__ == "__main__":
    main()